In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict

In [2]:
load_dotenv()

model = ChatGroq(model="llama-3.1-8b-instant")

In [3]:
class BlogState(TypedDict):
    question: str
    outline: str
    answer: str

In [4]:
def get_outline(state: BlogState) -> BlogState:
    question = state["question"]

    prompt = f"Give me the outlines for the blog on the given topic.\n {question}"

    outline = model.invoke(prompt).content

    state["outline"] = outline  # type:ignore

    return state


def get_blog(state: BlogState) -> BlogState:
    outline = state["outline"]

    prompt = f"Write a blog on the given outlines of the topic.\n {outline}"

    answer = model.invoke(prompt).content
    state["answer"] = answer  # type:ignore

    return state


In [5]:
graph = StateGraph(BlogState)

graph.add_node("get_outline", get_outline)
graph.add_node("get_blog", get_blog)

graph.add_edge(START, "get_outline")
graph.add_edge("get_outline", "get_blog")
graph.add_edge("get_blog", END)

workflow = graph.compile()

In [6]:
initial_state = {
    "question": "Why only hindu festival gets targetted like Diwali or Holi ?"
}

final_output = workflow.invoke(initial_state)

print(final_output)

{'question': 'Why only hindu festival gets targetted like Diwali or Holi ?', 'outline': 'Here\'s an outline for a blog on the topic "Why Only Hindu Festivals Get Targeted Like Diwali or Holi?"\n\n**I. Introduction**\n\n* Briefly introduce the topic and explain that Hindu festivals like Diwali and Holi have become increasingly targeted by extremists\n* Mention the purpose of the blog: to explore the reasons behind this targeting and the implications for Hindu communities\n\n**II. Historical Context: Hindu Festivals and Intolerance**\n\n* Discuss the history of Hindu festivals and their significance in Indian culture\n* Examine instances of Hindu festivals being targeted and vandalized in the past\n* Analyze how this targeting has contributed to a climate of intolerance and fear among Hindu communities\n\n**III. Stereotypes and Misconceptions About Hinduism**\n\n* Explore the stereotypes and misconceptions about Hinduism and its practices\n* Discuss how these stereotypes are often perpet